In [4]:
import pandas as pd
import numpy as np

def extract_and_filter_ibtracs(file_path):
    print("1. Loading IBTrACS Western Pacific Dataset...")
    df = pd.read_csv(file_path, skiprows=[1], low_memory=False, na_values=[' ', ''])
    
    print("2. Pruning Columns...")
    core_columns = [
        'SID', 'SEASON', 'ISO_TIME', 'NATURE', 'TRACK_TYPE',
        'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES',
        'USA_WIND', 'USA_PRES', 'DIST2LAND'
    ]
    df = df[core_columns].copy()
    
    print("3. Casting Data Types...")
    df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'])
    numeric_cols = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES', 
                    'USA_WIND', 'USA_PRES', 'DIST2LAND']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    print("4. Applying Meteorological Filters...")
    df = df[df['TRACK_TYPE'] == 'main']
    df = df[~df['NATURE'].isin(['ET', 'DS'])]
    
    print("5. Applying the 'DIST2LAND' Geospatial Shortcut...")
    df['is_over_land'] = df['DIST2LAND'] == 0
    
    print("6. Executing 6-Hour Temporal Resampling...")
    df = df.sort_values(['SID', 'ISO_TIME'])
    
    def resample_storm(storm_df):
        sid = storm_df['SID'].iloc[0]
        storm_df = storm_df.set_index('ISO_TIME')
        storm_df = storm_df.resample('6h').asfreq()
        storm_df.index.name = 'ISO_TIME'
        storm_df['SID'] = sid
        return storm_df
    
    df_6h = df.groupby('SID', group_keys=False).apply(resample_storm)
    df_6h = df_6h.reset_index()
    
    print("Data extraction complete.")
    return df_6h

In [5]:
# --- Execution ---
file_name = 'ibtracs.WP.list.v04r01.csv' 

# This line actually triggers the engine from Cell 1
processed_df = extract_and_filter_ibtracs(file_name)

# Display the results
print(f"\nFinal Shape: {processed_df.shape[0]} rows, {processed_df.shape[1]} columns")
display(processed_df.head(10))

1. Loading IBTrACS Western Pacific Dataset...
2. Pruning Columns...
3. Casting Data Types...
4. Applying Meteorological Filters...
5. Applying the 'DIST2LAND' Geospatial Shortcut...
6. Executing 6-Hour Temporal Resampling...
Data extraction complete.

Final Shape: 106662 rows, 13 columns


C:\Users\user\AppData\Local\Temp\ipykernel_22788\1267082598.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_6h = df.groupby('SID', group_keys=False).apply(resample_storm)


,ISO_TIME,SID,SEASON,NATURE,TRACK_TYPE,TOKYO_LAT,TOKYO_LON,TOKYO_WIND,TOKYO_PRES,USA_WIND,USA_PRES,DIST2LAND,is_over_land
0,1884-06-24 12:00:00,1884177N17124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1884-06-24 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,145.0,False
2,1884-06-25 00:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,77.0,False
3,1884-06-25 06:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,10.0,False
4,1884-06-25 12:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
5,1884-06-25 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
6,1884-06-26 00:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
7,1884-06-26 06:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,31.0,False
8,1884-06-26 12:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,90.0,False
9,1884-06-26 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,151.0,False


In [6]:
print("1. Slicing to the Modern Satellite Era (1980+)...")
# Drop pre-1980 storms where data fidelity is too low for ML
df_modern = processed_df[processed_df['SEASON'] >= 1980].copy()

print("2. Interpolating Small Trajectory Gaps...")
cols_to_interpolate = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']

# Using .transform() instead of .apply() ensures we don't lose the SID column
df_modern[cols_to_interpolate] = df_modern.groupby('SID')[cols_to_interpolate].transform(
    lambda x: x.interpolate(method='linear', limit=2, limit_direction='forward')
)

# Drop any rows that STILL have missing coordinates after interpolation
# (Meaning the storm dropped off radar for more than 12 hours)
df_clean = df_modern.dropna(subset=['TOKYO_LAT', 'TOKYO_LON']).copy()

print(f"Modern Era Shape: {df_clean.shape[0]} rows.")
display(df_clean[['SID', 'ISO_TIME', 'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']].head(10))

1. Slicing to the Modern Satellite Era (1980+)...
2. Interpolating Small Trajectory Gaps...
Modern Era Shape: 35412 rows.


,SID,ISO_TIME,TOKYO_LAT,TOKYO_LON,TOKYO_WIND,TOKYO_PRES
61125,1980094N02179,1980-04-05 00:00:00,9.5,180.0,NaN,1002.0
61126,1980094N02179,1980-04-05 06:00:00,10.4,178.6,NaN,1000.0
61127,1980094N02179,1980-04-05 12:00:00,11.8,177.7,40.0,994.0
61128,1980094N02179,1980-04-05 18:00:00,13.4,177.3,45.0,992.0
61129,1980094N02179,1980-04-06 00:00:00,14.8,177.2,45.0,992.0
61130,1980094N02179,1980-04-06 06:00:00,15.8,177.2,45.0,990.0
61131,1980094N02179,1980-04-06 12:00:00,16.7,177.6,60.0,985.0
61132,1980094N02179,1980-04-06 18:00:00,17.6,178.1,55.0,985.0
61133,1980094N02179,1980-04-07 00:00:00,18.5,178.9,45.0,990.0
61134,1980094N02179,1980-04-07 06:00:00,19.5,179.8,45.0,992.0


In [7]:
print("1. Preparing Physical Wind-Pressure Math....")
# Create a strict mask: Only calculate physics for storms over the ocean
mask_ocean = df_clean['is_over_land'] == False

print("2. Imputing Missing Pressure from Wind...")
# Equation: P_c = 1010 - (V_1min / 6.7) ** 1.5528
mask_missing_pres = df_clean['TOKYO_PRES'].isna() & df_clean['TOKYO_WIND'].notna() & mask_ocean
# Convert JMA 10-min wind to 1-min equivalent first
v_1min = df_clean.loc[mask_missing_pres, 'TOKYO_WIND'] * 1.14
df_clean.loc[mask_missing_pres, 'TOKYO_PRES'] = 1010 - (v_1min / 6.7) ** (1 / 0.644)

print("3. Imputing Missing Wind from Pressure...")
# Equation: V_1min = 6.7 * (1010 - P_c) ** 0.644
mask_missing_wind = df_clean['TOKYO_WIND'].isna() & df_clean['TOKYO_PRES'].notna() & mask_ocean

# Sub-mask A: Storms strong enough to have a pressure drop (P < 1010)
mask_calc_wind = mask_missing_wind & (df_clean['TOKYO_PRES'] < 1010)
v_1min_calc = 6.7 * ((1010 - df_clean.loc[mask_calc_wind, 'TOKYO_PRES']) ** 0.644)
# Convert 1-min back to JMA 10-min standard to maintain dataset consistency
df_clean.loc[mask_calc_wind, 'TOKYO_WIND'] = v_1min_calc / 1.14

# Sub-mask B: Weak baseline depressions (P >= 1010)
mask_base_wind = mask_missing_wind & (df_clean['TOKYO_PRES'] >= 1010)
df_clean.loc[mask_base_wind, 'TOKYO_WIND'] = 15.0 # Set to baseline tropical disturbance wind (15 knots)

print("4. Dropping Unrecoverable Rows...")
# Drop any rows where BOTH wind and pressure were missing, or it was missing over land
df_clean = df_clean.dropna(subset=['TOKYO_WIND', 'TOKYO_PRES']).copy()

print(f"Post-Imputation Shape: {df_clean.shape[0]} rows.")
display(df_clean[['SID', 'ISO_TIME', 'TOKYO_WIND', 'TOKYO_PRES', 'is_over_land']].head(10))

1. Preparing Physical Wind-Pressure Math....
2. Imputing Missing Pressure from Wind...
3. Imputing Missing Wind from Pressure...
4. Dropping Unrecoverable Rows...
Post-Imputation Shape: 34391 rows.


,SID,ISO_TIME,TOKYO_WIND,TOKYO_PRES,is_over_land
61125,1980094N02179,1980-04-05 00:00:00,22.426418,1002.0,False
61126,1980094N02179,1980-04-05 06:00:00,25.892260,1000.0,False
61127,1980094N02179,1980-04-05 12:00:00,40.000000,994.0,False
61128,1980094N02179,1980-04-05 18:00:00,45.000000,992.0,False
61129,1980094N02179,1980-04-06 00:00:00,45.000000,992.0,False
61130,1980094N02179,1980-04-06 06:00:00,45.000000,990.0,False
61131,1980094N02179,1980-04-06 12:00:00,60.000000,985.0,False
61132,1980094N02179,1980-04-06 18:00:00,55.000000,985.0,False
61133,1980094N02179,1980-04-07 00:00:00,45.000000,990.0,False
61134,1980094N02179,1980-04-07 06:00:00,45.000000,992.0,False


In [8]:
print("1. Calculating Seasonal Physics (Sine/Cosine)...")
# Extract day of the year and convert to a cyclical wave
# This tells the AI the difference between a July storm and a December storm
df_clean['day_of_year'] = df_clean['ISO_TIME'].dt.dayofyear
df_clean['sin_season'] = np.sin(2 * np.pi * df_clean['day_of_year'] / 365.25)
df_clean['cos_season'] = np.cos(2 * np.pi * df_clean['day_of_year'] / 365.25)

print("2. Calculating Velocity Differentials...")
# Ensure chronological order
df_clean = df_clean.sort_values(by=['SID', 'ISO_TIME'])

# How much did the storm move in the last 6 hours? (Momentum)
df_clean['delta_lat'] = df_clean.groupby('SID')['TOKYO_LAT'].diff()
df_clean['delta_lon'] = df_clean.groupby('SID')['TOKYO_LON'].diff()

print("3. Building the History Sliding Window (t-6 and t-12)...")
cols_to_shift = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']

for col in cols_to_shift:
    # .shift(1) pulls the value from 6 hours ago
    df_clean[f'{col}_t-6'] = df_clean.groupby('SID')[col].shift(1)
    # .shift(2) pulls the value from 12 hours ago
    df_clean[f'{col}_t-12'] = df_clean.groupby('SID')[col].shift(2)

print("4. Formulating the Predictive Targets (t+6)...")
# .shift(-1) looks into the FUTURE to get the "Answer" the AI must guess
df_clean['TARGET_LAT'] = df_clean.groupby('SID')['TOKYO_LAT'].shift(-1)
df_clean['TARGET_LON'] = df_clean.groupby('SID')['TOKYO_LON'].shift(-1)

print("5. Dropping Incomplete Windows...")
# Shifting creates NaNs (e.g., the first 12 hours of a storm have no history, 
# and the last 6 hours have no future target). We must drop these incomplete rows.
df_ml = df_clean.dropna().copy()

print(f"Final Machine Learning Dataset Shape: {df_ml.shape[0]} rows.")
display(df_ml[['SID', 'ISO_TIME', 'TOKYO_LAT_t-12', 'TOKYO_LAT_t-6', 'TOKYO_LAT', 'TARGET_LAT']].head(5))

1. Calculating Seasonal Physics (Sine/Cosine)...
2. Calculating Velocity Differentials...
3. Building the History Sliding Window (t-6 and t-12)...
4. Formulating the Predictive Targets (t+6)...
5. Dropping Incomplete Windows...
Final Machine Learning Dataset Shape: 14476 rows.


,SID,ISO_TIME,TOKYO_LAT_t-12,TOKYO_LAT_t-6,TOKYO_LAT,TARGET_LAT
86452,2001125N05129,2001-05-10 00:00:00,11.8,12.3,13.1,14.1
86453,2001125N05129,2001-05-10 06:00:00,12.3,13.1,14.1,15.3
86454,2001125N05129,2001-05-10 12:00:00,13.1,14.1,15.3,16.2
86455,2001125N05129,2001-05-10 18:00:00,14.1,15.3,16.2,17.1
86456,2001125N05129,2001-05-11 00:00:00,15.3,16.2,17.1,17.6
